# Registro histórico: 04_nova_calibration

Origem: `B01` — `02-NOVAcreateexcel.ipynb`.

Variante com ordenação natural, filtro de potencial e R².

**Limitações conhecidas:** Depende de planilhas externas; escreve cabeçalhos sobre a primeira linha de dados da tabela de máximos e captura erros sem detalhar.

Cópia organizada em 25/10/2026. Código e texto originais preservados abaixo; saídas, contagens de execução e metadados do Colab removidos. As métricas salvas estão em `results/historical_evidence.json`. Não executar em lote: estes cadernos dependem do ambiente e de arquivos originais. Veja `docs/experiments.md` para o fluxo novo.


In [ ]:
import pandas as pd
from openpyxl import Workbook
from glob import glob
import os
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline, BSpline
import numpy as np
from scipy.signal import argrelextrema
import re

In [ ]:
def key_func(s):
        parts = re.findall(r'(\D+|\d+)', s)
        return [int(part) if part.isdigit() else part for part in parts]

def NOVAsheet(folder, newpath, newlinepath):
    arcs = sorted(glob(folder + '/*.xlsx'), key=key_func)

    pot = 'Potential applied (V)'
    delta = 'WE(1).δ.Current (A)'
    # Carrega a planilha existente
    wb = Workbook()

    # Seleciona a planilha ativa
    planilha = wb.active
    planilha.append([pot])
    wb.save(newpath)
    newdata = pd.read_excel(newpath)
    df = pd.read_excel(arcs[0])
    newdata[pot] = df[pot]
    count = 1
    for arq in arcs:
        try:
            df = pd.read_excel(arq)
            newdata = newdata.assign(b = df[delta])
            newdata.rename(columns={'b': os.path.splitext(os.path.basename(arq))[0]}, inplace=True)
        except:
            print('erro')
        finally:
            count = count + 1
    
    newdata = newdata[newdata['Potential applied (V)'] >= -0.025177002]

    newdata.to_excel(newpath, index=False)



    pattern = re.compile(r'(\w+)-(\d+)(\w+)')


    wb1 = Workbook()

    # Seleciona a planilha ativa
    planilha = wb1.active

    match = pattern.match(newdata.columns[1])
    planilha.append([f'{match.group(1)} ({match.group(3)})'])
    planilha.append(['Corrente máxima (A)'])

    for i in range(1, newdata.shape[1]):
        maior = max(newdata.iloc[:, i].values)
        name = newdata.columns[i]
        match = pattern.match(name)
        planilha[f'A{i}'] = int(match.group(2))
        planilha[f'B{i}'] = maior

    wb1.save(newlinepath)



In [ ]:
def plotagrafico(arquivo):
    # Lê o arquivo do Excel
    df = pd.read_excel(arquivo)

    # Obtém os valores do eixo x da primeira coluna
    x = df.iloc[:, 0].values

    # Itera sobre as outras colunas
    for i in range(1, df.shape[1]):
        y = df.iloc[:, i].values

        # Cria uma função de interpolação suavizada
        spl = make_interp_spline(x, y, k=3)
        xnew = np.linspace(x.min(), x.max(), 500)
        y_smooth = spl(xnew)

        # Plota a curva suavizada
        plt.plot(xnew, y_smooth, label=df.columns[i])

    plt.ylim(4.5*(10**-5), 5.5*(10**-5))
    plt.xlim(-0.025177, 0.4)
    plt.xlabel('Potential applied (V)')
    plt.ylabel('WE(1).δ.Current (A)')

    # Adiciona uma legenda
    plt.legend()

    # Mostra o gráfico
    plt.show()

In [ ]:
def linesheet(arquivo):
    df = pd.read_excel(arquivo)

    x = []
    y = []
    for i in range(1, df.shape[1]):
        yi = max(df.iloc[:, i].values)
        name = df.columns[i]
        
        pattern = re.compile(r'(\w+)-(\d+)(\w+)')

        # Usa a expressão regular para extrair as partes da string

        match = pattern.match(name)
        
        xi = int(match.group(2))

        x.append(xi)
        y.append(yi)
        plt.scatter(xi, yi, label=name)

    # Realizando a regressão linear para encontrar a linha de tendência
    x = np.array(x)
    y = np.array(y)
    m, b = np.polyfit(x, y, 1)  # ajuste linear dos dados

    # Calculando o R²
    y_pred = m * x + b
    ss_total = np.sum((y - np.mean(y))**2)
    ss_residual = np.sum((y - y_pred)**2)
    r_squared = 1 - (ss_residual / ss_total)

    plt.ylabel('Corrente máxima (A)')
    plt.xlabel(f'Concentração da amostra de {match.group(1)} ({match.group(3)})')
    # Formatando os coeficientes para micro ou nano, conforme necessário

    equation = f'Equação da reta: y = {m:.2e}x + {b:.2e}'
    print(equation)
    print(f'R² = {r_squared:.3f}')

    plt.plot(x, m*x + b, color='red', label='Linha de tendência')
        

In [ ]:
df = pd.read_excel('tesss2/novaplanilha.xlsx')

# Valor de referência
valor_referencia = -0.025177002


# Remove as linhas onde o valor na coluna 'A' é menor que o valor de referência
df_filtrado = df[df['Potential applied (V)'] >= valor_referencia]

# Salva o DataFrame filtrado em uma nova planilha
df_filtrado.to_excel('tesss2/dados_filtrados.xlsx', index=False)

In [ ]:
NOVAsheet('planilhasNOVA', 'tesss2/novaplanilha.xlsx', 'tesss2/novaplanilha2.xlsx')

In [ ]:
linesheet('tesss2/novaplanilha.xlsx')

In [ ]:
plotagrafico('tesss2/novaplanilha.xlsx')
